# EDA — Credit Card Fraud Detection

> **Dataset**: [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — ULB Machine Learning Group  
> **Registros**: 284.807 transações | **Período**: 2 dias (setembro 2013)  
> **Features**: Time, Amount + V1–V28 (componentes PCA dos dados originais)

---

## Objetivo

Entender a estrutura do dataset, identificar padrões de fraude e embasar decisões de feature engineering antes do treinamento.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_PATH = Path("../data/raw/creditcard.csv")

if not DATA_PATH.exists():
    print("⚠️  Dataset não encontrado. Execute:")
    print("    make data   (requer KAGGLE_USERNAME e KAGGLE_KEY no .env)")
    print()
    print("Gerando dados sintéticos para demonstração da estrutura do EDA...")
    from faker import Faker
    rng = np.random.default_rng(42)
    n = 10_000
    fraud_mask = rng.random(n) < 0.0017
    df = pd.DataFrame({
        "Time": rng.uniform(0, 172792, n),
        "Amount": np.where(fraud_mask, rng.exponential(200, n), rng.exponential(80, n)),
        **{f"V{i}": rng.standard_normal(n) * (2 if fraud_mask.any() and i in [14, 17] else 1)
           for i in range(1, 29)},
        "Class": fraud_mask.astype(int),
    })
    print(f"✅ Dataset sintético gerado: {len(df):,} registros ({fraud_mask.sum()} fraudes)")
else:
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Dataset real carregado: {len(df):,} registros")

print(df.shape)

## 1. Visão Geral do Dataset

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Nenhum valor nulo esperado — verificação de contrato
nulls = df.isnull().sum()
assert nulls.sum() == 0, f"Valores nulos encontrados: {nulls[nulls > 0]}"
print("✅ Nenhum valor nulo — contrato de qualidade OK")

In [ ]:
df.describe().T.style.background_gradient(cmap="Blues", subset=["mean", "std"])

## 2. Desbalanceamento de Classes

> **Insight principal**: apenas 0.17% das transações são fraudes. Este é o maior desafio do problema — um modelo que sempre prediz "legítima" teria 99.83% de acurácia mas 0% de recall em fraudes.

In [ ]:
class_counts = df["Class"].value_counts()
fraud_pct = df["Class"].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contagem absoluta
axes[0].bar(["Legítima (0)", "Fraude (1)"], class_counts.values,
            color=["#4CAF50", "#f44336"], edgecolor="white")
axes[0].set_title("Distribuição de Classes — Escala Absoluta")
axes[0].set_ylabel("Contagem")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 100, f"{v:,}", ha="center", fontweight="bold")

# Proporção
axes[1].pie([100 - fraud_pct, fraud_pct],
            labels=[f"Legítima\n{100-fraud_pct:.2f}%", f"Fraude\n{fraud_pct:.2f}%"],
            colors=["#4CAF50", "#f44336"],
            startangle=90, autopct="%1.2f%%")
axes[1].set_title("Proporção de Classes")

plt.suptitle("Desbalanceamento Extremo — Mitigação: class_weight='balanced' + pos_weight",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nRatio fraude/legítima: 1:{int((1/fraud_pct*100)):,}")
print(f"Implicação: usar AUC-ROC + F1 + Precision/Recall em vez de acurácia")

## 3. Distribuição Temporal

> **Insight**: `Time` registra segundos desde a primeira transação. Convertemos para `Hour` (0–24) para capturar padrões circadianos — fraudes têm distribuição diferente por hora do dia.

In [ ]:
df["Hour"] = (df["Time"] / 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribuição geral por hora
axes[0].hist(df["Hour"], bins=48, color="#2196F3", alpha=0.7, edgecolor="white")
axes[0].set_title("Volume de Transações por Hora do Dia")
axes[0].set_xlabel("Hora")
axes[0].set_ylabel("Transações")

# Legítimas vs fraudes por hora
legit = df[df["Class"] == 0]["Hour"]
fraud = df[df["Class"] == 1]["Hour"]
axes[1].hist(legit, bins=48, alpha=0.6, label="Legítima", color="#4CAF50", density=True)
axes[1].hist(fraud, bins=48, alpha=0.7, label="Fraude", color="#f44336", density=True)
axes[1].set_title("Distribuição Normalizada por Hora — Legítima vs Fraude")
axes[1].set_xlabel("Hora")
axes[1].legend()

plt.tight_layout()
plt.show()

# Risco por faixa horária
df["hour_bin"] = pd.cut(df["Hour"], bins=[0,6,12,18,24], labels=["00-06","06-12","12-18","18-24"])
risk_by_hour = df.groupby("hour_bin", observed=True)["Class"].mean() * 100
print("\nRisco de fraude por faixa horária (%)")
print(risk_by_hour.to_string())
df.drop(columns=["hour_bin"], inplace=True)

## 4. Distribuição de Amount

> **Insight**: `Amount` tem distribuição muito assimétrica. Aplicamos `log(Amount + 1)` para normalizar. Fraudes têm padrão de valor diferente das transações legítimas.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Amount bruto
axes[0,0].hist(df[df["Class"]==0]["Amount"], bins=100, color="#4CAF50", alpha=0.7, label="Legítima")
axes[0,0].hist(df[df["Class"]==1]["Amount"], bins=100, color="#f44336", alpha=0.7, label="Fraude")
axes[0,0].set_title("Amount — Escala Original")
axes[0,0].set_xlabel("Amount")
axes[0,0].legend()
axes[0,0].set_xlim(0, 1000)

# Amount log-transformado
df["Amount_log"] = np.log1p(df["Amount"])
axes[0,1].hist(df[df["Class"]==0]["Amount_log"], bins=50, color="#4CAF50", alpha=0.7, label="Legítima")
axes[0,1].hist(df[df["Class"]==1]["Amount_log"], bins=50, color="#f44336", alpha=0.7, label="Fraude")
axes[0,1].set_title("Amount — Log(Amount + 1)  ✅ Feature Engineering")
axes[0,1].set_xlabel("log(Amount + 1)")
axes[0,1].legend()

# Boxplot Amount por classe
df.boxplot(column="Amount", by="Class", ax=axes[1,0], notch=True)
axes[1,0].set_title("Boxplot Amount por Classe")
axes[1,0].set_xlabel("Classe (0=Legítima, 1=Fraude)")
axes[1,0].set_ylim(0, 2000)

# Estatísticas Amount
amount_stats = df.groupby("Class")["Amount"].agg(["mean", "median", "std", "max"])
amount_stats.index = ["Legítima", "Fraude"]
axes[1,1].axis("off")
table_data = [[f"{v:.2f}" for v in row] for row in amount_stats.values]
tbl = axes[1,1].table(
    cellText=table_data,
    rowLabels=amount_stats.index,
    colLabels=["Média", "Mediana", "Desvio", "Máximo"],
    loc="center", cellLoc="center"
)
tbl.scale(1, 2)
axes[1,1].set_title("Estatísticas Amount por Classe")

plt.suptitle("Análise de Amount — Fraudes tendem a valores menores e mais concentrados", y=1.01)
plt.tight_layout()
plt.show()

df.drop(columns=["Amount_log"], inplace=True)

## 5. Features PCA (V1–V28) — Poder Discriminativo

> **Insight**: As features V1–V28 são componentes PCA dos dados originais (anonimizados por LGPD). Algumas têm alto poder discriminativo — especialmente **V14, V17, V12** que aparecem consistentemente em análises de importância para fraude.

In [ ]:
from scipy import stats

v_cols = [f"V{i}" for i in range(1, 29)]
fraud_df = df[df["Class"] == 1]
legit_df = df[df["Class"] == 0]

# Diferença de médias normalizada (Cohen's d simplificado)
discriminative = []
for col in v_cols:
    t_stat, p_val = stats.ttest_ind(fraud_df[col], legit_df[col], equal_var=False)
    mean_diff = abs(fraud_df[col].mean() - legit_df[col].mean())
    pooled_std = np.sqrt((fraud_df[col].std()**2 + legit_df[col].std()**2) / 2)
    cohen_d = mean_diff / pooled_std if pooled_std > 0 else 0
    discriminative.append({"feature": col, "cohen_d": cohen_d, "p_value": p_val})

discrim_df = pd.DataFrame(discriminative).sort_values("cohen_d", ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#f44336" if d > 0.5 else "#FF9800" if d > 0.2 else "#4CAF50"
          for d in discrim_df["cohen_d"]]
ax.bar(discrim_df["feature"], discrim_df["cohen_d"], color=colors, edgecolor="white")
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.7, label="Alto (>0.5)")
ax.axhline(y=0.2, color="orange", linestyle="--", alpha=0.7, label="Médio (>0.2)")
ax.set_title("Poder Discriminativo das Features V1–V28 (Cohen's d)")
ax.set_xlabel("Feature")
ax.set_ylabel("Cohen's d (|média fraude − média legítima| / desvio pooled)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

top5 = discrim_df.head(5)
print("Top 5 features mais discriminativas:")
print(top5[["feature", "cohen_d", "p_value"]].to_string(index=False))

In [ ]:
# Distribuição das top 4 features discriminativas por classe
top4_features = discrim_df.head(4)["feature"].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top4_features):
    axes[i].hist(legit_df[feat], bins=60, alpha=0.6, label="Legítima",
                 color="#4CAF50", density=True)
    axes[i].hist(fraud_df[feat], bins=60, alpha=0.7, label="Fraude",
                 color="#f44336", density=True)
    axes[i].set_title(f"{feat} — Cohen's d = {discrim_df.loc[discrim_df['feature']==feat, 'cohen_d'].values[0]:.3f}")
    axes[i].legend()
    axes[i].set_xlabel(feat)

plt.suptitle("Top 4 Features com Maior Poder Discriminativo\n"
             "Essas features são as mais importantes para o modelo e para o SHAP", y=1.02)
plt.tight_layout()
plt.show()

## 6. Matriz de Correlação

> **Insight**: Por serem componentes PCA, as features V1–V28 são ortogonais entre si (correlação próxima de zero). Isso é esperado e benéfico — sem multicolinearidade.

In [ ]:
# Correlação com a variável alvo
corr_with_target = df[v_cols + ["Amount", "Time", "Class"]].corr()["Class"].drop("Class").sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Correlação com Class
colors = ["#f44336" if v < -0.1 or v > 0.1 else "#90A4AE" for v in corr_with_target]
axes[0].barh(corr_with_target.index, corr_with_target.values, color=colors)
axes[0].axvline(x=0.1, color="orange", linestyle="--", alpha=0.7)
axes[0].axvline(x=-0.1, color="orange", linestyle="--", alpha=0.7)
axes[0].set_title("Correlação de Pearson com Class (alvo)")
axes[0].set_xlabel("Correlação")

# Heatmap subset das mais correlacionadas
top_corr = corr_with_target.abs().nlargest(10).index.tolist() + ["Class"]
corr_matrix = df[top_corr].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, ax=axes[1], annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, mask=mask,
            annot_kws={"size": 8})
axes[1].set_title("Correlação — Top 10 Features + Class")

plt.tight_layout()
plt.show()

## 7. Drift Temporal — Estabilidade das Features

> **Insight**: O dataset cobre 2 dias. Dividir por período (dia 1 vs dia 2) simula o cenário de drift que o Evidently monitorará em produção. Permite calibrar os thresholds PSI.

In [ ]:
# Simula drift: divide dataset em dois períodos
midpoint = df["Time"].median()
period1 = df[df["Time"] <= midpoint]
period2 = df[df["Time"] > midpoint]

print(f"Período 1: {len(period1):,} transações | Fraudes: {period1['Class'].sum():,} ({period1['Class'].mean()*100:.3f}%)")
print(f"Período 2: {len(period2):,} transações | Fraudes: {period2['Class'].sum():,} ({period2['Class'].mean()*100:.3f}%)")

# PSI manual para as top 5 features
def compute_psi(ref, cur, bins=10):
    ref_pct, edges = np.histogram(ref, bins=bins, density=True)
    cur_pct, _ = np.histogram(cur, bins=edges, density=True)
    ref_pct = ref_pct + 1e-6
    cur_pct = cur_pct + 1e-6
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

psi_results = {}
for feat in v_cols + ["Amount"]:
    psi_results[feat] = abs(compute_psi(period1[feat], period2[feat]))

psi_series = pd.Series(psi_results).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 4))
colors = ["#f44336" if v > 0.2 else "#FF9800" if v > 0.1 else "#4CAF50" for v in psi_series]
ax.bar(psi_series.index, psi_series.values, color=colors, edgecolor="white")
ax.axhline(y=0.2, color="red", linestyle="--", label="Retrain trigger (PSI > 0.2)")
ax.axhline(y=0.1, color="orange", linestyle="--", label="Warning (PSI > 0.1)")
ax.set_title("PSI entre Período 1 e Período 2 — Estabilidade das Features")
ax.set_ylabel("PSI")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nFeatures com potencial drift (PSI > 0.1):")
print(psi_series[psi_series > 0.1].to_string())

## 8. Decisões de Feature Engineering

Com base no EDA, as seguintes transformações foram implementadas em `src/features/feature_engineering.py`:

In [ ]:
decisions = {
    "Amount_scaled": "StandardScaler — normaliza distribuição assimétrica do valor",
    "Time_scaled":   "StandardScaler — normaliza time em segundos",
    "Hour":          "Time / 3600 % 24 — captura padrão circadiano de fraude",
    "Amount_log":    "log(Amount + 1) — reduz assimetria, aproxima distribuição normal",
    "V1–V28":        "Mantidos sem transformação — já são componentes PCA ortogonais",
    "Drop Time":     "Removido após criar Hour — não agrega informação adicional",
    "Drop Amount":   "Removido após criar Amount_scaled e Amount_log",
}

print("Feature Engineering — Decisões baseadas no EDA")
print("=" * 65)
for feat, reason in decisions.items():
    print(f"  {feat:<20} → {reason}")

print("\nEsquema de features de saída: 32 colunas")
print("  Amount_scaled, Time_scaled, Hour, Amount_log, V1–V28")
print("  Validadas via Pandera DataFrameSchema em feature_engineering.py")

## 9. Métricas de Negócio → Métricas Técnicas

> Esta seção documenta como as métricas de negócio se traduzem em KPIs técnicos — item obrigatório no Model Card.

In [ ]:
mapping = [
    ("Minimizar fraudes não detectadas",
     "Recall alto (≥ 0.80)",
     "Uma fraude não detectada = prejuízo financeiro direto ao cliente"),
    ("Minimizar falsos positivos",
     "Precision razoável (≥ 0.85)",
     "Bloquear transações legítimas gera atrito e perda de cliente"),
    ("Separar bem as classes",
     "AUC-ROC ≥ 0.95",
     "Permite ajustar threshold sem retreinar o modelo"),
    ("Equilíbrio recall/precision",
     "F1 ≥ 0.82",
     "KPI único para comparação de versões no champion-challenger"),
    ("Resposta em tempo real",
     "Latência p99 < 200ms",
     "Transação não pode ser bloqueada por timeout da API"),
]

business_df = pd.DataFrame(mapping,
    columns=["Objetivo de Negócio", "Métrica Técnica", "Justificativa"])
business_df.index = range(1, len(business_df) + 1)
business_df.style.set_properties(**{"text-align": "left"})

## 10. Sumário dos Insights para o Modelo

| # | Insight | Decisão no Pipeline |
|---|---|---|
| 1 | Desbalanceamento extremo (0.17% fraudes) | `class_weight='balanced'` + `pos_weight` no PyTorch |
| 2 | Fraudes têm padrão horário diferente | Feature `Hour` criada |
| 3 | Amount tem distribuição assimétrica | `Amount_log` + `Amount_scaled` |
| 4 | V14, V17, V12 têm alto poder discriminativo | SHAP os confirmará como top features |
| 5 | Features PCA são ortogonais (sem multicolinearidade) | Sem necessidade de seleção de features |
| 6 | Drift entre períodos existe — Amount e algumas V-features | PSI monitorado via Evidently, threshold 0.1/0.2 |
| 7 | Métricas de acurácia são enganosas | Usamos AUC-ROC, F1, Precision, Recall |